## Lecture 4

In this notebook we implement a **Wasserstein GAN (WGAN)** on the MNIST dataset. Instead of the usual BCE / KL based GAN loss, the Discriminator (a.k.a. *critic*) is trained to estimate the **Earth-Mover (Wasserstein-1) distance** between the real and the generated distributions. Training progress (losses, gradients, model graphs and generated images) is logged with **TensorBoard**.

**References**
- PyTorch TensorBoard docs: https://pytorch.org/docs/stable/tensorboard.html
- Maxout activation source: https://github.com/pytorch/pytorch/issues/805
- Original lecture Colab notebook: https://colab.research.google.com/drive/13oGV5GNcnZVKw1zzsAEAH9nvms04uYnT
- Original WGAN paper code: https://github.com/martinarjovsky/WassersteinGAN
- Wasserstein GAN paper (Arjovsky, Chintala, Bottou, 2017): https://arxiv.org/abs/1701.07875

In [ ]:
# Original lecture used: ! pip3 install tensorboard future
# `%pip` installs into the currently selected kernel (ai-ml-ds) instead of the system Python.
%pip install tensorboard future

In [ ]:
import torch
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:
# Tensorboard
# https://pytorch.org/docs/stable/tensorboard.html
from torch.utils.tensorboard import SummaryWriter


# Writer will output to ./runs/ directory by default
tb_writer = SummaryWriter()

tb_writer.add_text(
    'WGAN',
    'Init',
    0
)

In [ ]:
torch.cuda.is_available(), torch.backends.mps.is_available()

In [ ]:
# Device selection adapted to run locally on macOS with Apple GPU (MPS) support,
# falling back to CUDA (if available) and finally CPU.
device = torch.device('cpu')
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')


device

In [ ]:
noise_dim = 100

Generator Model

In [ ]:
class Generator(torch.nn.Module):

    def __init__(self):

        super(Generator, self).__init__()

        self.fcn = torch.nn.Sequential(
            # Fully Connected Layer 1
            torch.nn.Linear(
                in_features=noise_dim,
                out_features=240,
                bias=True
            ),
            torch.nn.ReLU(),
            torch.nn.Dropout(),
            # Fully Connected Layer 2
            torch.nn.Linear(
                in_features=240,
                out_features=240,
                bias=True
            ),
            torch.nn.ReLU(),
            torch.nn.Dropout(),
            # Fully Connected Layer 3
            torch.nn.Linear(
                in_features=240,
                out_features=240,
                bias=True
            ),
            torch.nn.ReLU(),
            torch.nn.Dropout(),
            # Fully Connected Layer 4
            torch.nn.Linear(
                in_features=240,
                out_features=784,
                bias=True
            ),
            torch.nn.Sigmoid()
        )

    def forward(self, batch):
        ret = batch.view(batch.size(0), -1)
        ret = self.fcn(ret)
        return ret

Maxout Activation

***Source: https://github.com/pytorch/pytorch/issues/805***

In [ ]:
class Maxout(torch.nn.Module):

    def __init__(self, num_pieces):

        super(Maxout, self).__init__()

        self.num_pieces = num_pieces

    def forward(self, x):

        # x.shape = (batch_size? x 625)

        assert x.shape[1] % self.num_pieces == 0  # 625 % 5 = 0

        ret = x.view(
            *x.shape[:1],  # batch_size
            x.shape[1] // self.num_pieces,  # piece-wise linear
            self.num_pieces,  # num_pieces
            *x.shape[2:]  # remaining dimensions if any
        )

        # ret.shape = (batch_size? x 125 x 5)

        # https://pytorch.org/docs/stable/torch.html#torch.max
        ret, _ = ret.max(dim=2)

        # ret.shape = (batch_size? x 125)

        return ret

Discriminator Model

Unlike the vanilla GAN, the WGAN critic has **no Sigmoid** at the output layer: the last layer is linear and the forward pass returns the mean score over the batch as a single scalar value of one dimension (`outputs.view(1)`).

In [ ]:
class Discriminator(torch.nn.Module):

    def __init__(self):

        super(Discriminator, self).__init__()

        self.fcn = torch.nn.Sequential(
            # Fully Connected Layer 1
            torch.nn.Linear(
                in_features=784,
                out_features=240,
                bias=True
            ),
            Maxout(5),
            # Fully Connected Layer 2
            torch.nn.Linear(
                in_features=48,
                out_features=240,
                bias=True
            ),
            Maxout(5),
            # Fully Connected Layer 3
            torch.nn.Linear(
                in_features=48,
                out_features=1,
                bias=True
            )
        )

    def forward(self, batch):
        inputs = batch.view(batch.size(0), -1)
        outputs = self.fcn(inputs)
        outputs = outputs.mean(0)
        return outputs.view(1)

MNIST Dataset

We reuse the local dataset folder `./data/mnist` (already used by the previous notebooks).

In [ ]:
import torchvision

In [ ]:
class FlattenTransform:

    def __call__(self, inputs):
        return inputs.view(inputs.shape[0], -1)


data_train = torchvision.datasets.MNIST(
    './data/mnist',
    train=True,
    download=True,
    transform=torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        FlattenTransform()
    ])
)

In [ ]:
BATCH_SIZE = 64

# num_workers=4 in the original lecture. On macOS, DataLoader workers use the `spawn`
# start method and cannot pickle classes defined inside a notebook (FlattenTransform),
# so we load in the main process (num_workers=0) to avoid the worker crash.
train_loader = torch.utils.data.DataLoader(
    data_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

GAN Optimizer

As advised by the authors of the WGAN paper, we use **RMSprop** (no momentum) with a small learning rate of `0.00005` for both networks.

In [ ]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)


discriminator_optimizer = torch.optim.RMSprop(
    discriminator.parameters(),
    lr=0.00005,
    momentum=0
)

generator_optimizer = torch.optim.RMSprop(
    generator.parameters(),
    lr=0.00005,
    momentum=0
)

Earth-Mover distance

In [ ]:
is_emd = True  # switch between KL & EMD

In [ ]:
criterion = torch.nn.KLDivLoss()

Visualize Output

In [ ]:
def visualizeGAN(tgt_pth, images, epoch):

    fig, axes = plt.subplots(2, 5, figsize=(20, 18))

    fig.suptitle('Epoch {}'.format(str(epoch).zfill(4)))

    for row, axe in enumerate(axes):
        for col, cell in enumerate(axe):
            cell.imshow(
                images[row * 5 + col].squeeze(),  # (1, 28, 28) -> (28, 28) for imshow
                cmap='gray'
            )

            cell.axis("off")


    plt.axis("off")
    plt.tight_layout()

    fig.savefig(os.path.join(tgt_pth, '{}.jpg'.format(str(epoch).zfill(4))))

    plt.close()

In [ ]:
import os

visuals_dir = 'visual-setion-3-lecture-4'

if not os.path.exists(visuals_dir):
    os.mkdir(visuals_dir)

GAN Training

With the Earth-Mover distance the critic output is a scalar of shape `(1,)` (mean over the batch), so the gradients passed to `.backward()` must also have shape `(1,)`: `+1` for real and `-1` for fake (the original lecture used `(BATCH_SIZE, 1)`, which raises a shape-mismatch error in current PyTorch versions).

In [ ]:
real_labels = torch.ones(1).to(device)  # originally torch.ones(BATCH_SIZE, 1)
fake_labels = ( -1 * torch.ones(1) ).to(device)  # originally -1 * torch.ones(BATCH_SIZE, 1)

test_set = torch.randn(25, noise_dim).to(device)

num_epochs = 1024  # @todo while not converged...
num_steps = len(train_loader) // BATCH_SIZE

Training loop: for every generator update the critic is trained 5 times and its weights are clamped to the cube `[-0.01, 0.01]` (weight clipping, as in the paper). Losses and gradients are logged to TensorBoard.

> Note: the original lecture called `tb_writer.add_graph(...)` on every iteration. Tracing the models is slow and only needs to happen once, so the graphs are logged only in the very first iteration (`log_graph`). A `TracerWarning` about converting a tensor to a Python boolean (from the `assert` in `Maxout`) is expected and harmless.

In [ ]:
for epoch in range(num_epochs):

    for i, (images, _) in enumerate(train_loader):

        if i == num_steps:
            break

        # the model graphs only need to be traced/logged once
        log_graph = epoch == 0 and i == 0

        # Train Discriminator
        discriminator_loss = 0
        for k in range(5):

            real_images = images.to(device)

            fake_images = generator(
                torch.randn(BATCH_SIZE, noise_dim).to(device)
            )

            # clamp parameters to a cube
            for p in discriminator.parameters():
                p.data.clamp_(-0.01, 0.01)

            discriminator_optimizer.zero_grad()

            real_outputs = discriminator(real_images)
            fake_outputs = discriminator(fake_images)

            if is_emd:
                # perform Earth-Mover
                real_outputs.backward(real_labels)
                fake_outputs.backward(fake_labels)

                discriminator_loss += (real_outputs - fake_outputs).detach()
            else:
                # perform criterion
                d_x = criterion(real_outputs, real_labels)
                d_g_z = criterion(fake_outputs, fake_labels)

                d_x.backward()
                d_g_z.backward()

                discriminator_loss += (d_x + d_g_z).detach()

            tb_writer.add_scalar(
                'discriminator/gradients',
                discriminator.fcn[-1].weight.grad.abs().mean().item(),
                int('{}{}'.format(epoch, k))
            )

            if log_graph and k == 0:
                tb_writer.add_graph(
                    discriminator,
                    real_images
                )

                tb_writer.add_graph(
                    discriminator,
                    fake_images
                )

            discriminator_optimizer.step()

        # log loss for criterion
        tb_writer.add_scalar(
            'discriminator/loss',
            discriminator_loss.item() / 5,
            epoch
        )


        # Train Generator
        z = torch.randn(BATCH_SIZE, noise_dim).to(device)

        generator.zero_grad()

        outputs = discriminator(generator(z))

        if is_emd:
            # perform Earth-Mover
            outputs.backward(real_labels)

            tb_writer.add_scalar(
                'generator/loss',
                -1 * outputs.item(),
                epoch
            )
        else:
            # perform criterion
            loss = criterion(outputs, real_labels)

            loss.backward()

            tb_writer.add_scalar(
                'generator/loss',
                loss.item(),
                epoch
            )

        tb_writer.add_scalar(
            'generator/gradients',
            generator.fcn[-2].weight.grad.abs().mean().item(),
            epoch
        )

        if log_graph:
            tb_writer.add_graph(
                generator,
                z
            )

        generator_optimizer.step()



    # Visualize Results
    if epoch % 10 == 0:

        generated = generator(test_set).detach().cpu().view(-1, 1, 28, 28)

#         visualizeGAN(visuals_dir, generated, epoch)

        grid = torchvision.utils.make_grid(
            generated,
            nrow=5,
            padding=10,
            pad_value=1
        )

        tb_writer.add_image('generator/outputs', grid, epoch)

tb_writer.flush()

In [ ]:
# Visualize Results
generated = generator(test_set).detach().cpu().view(-1, 1, 28, 28)

grid = torchvision.utils.make_grid(
    generated,
    nrow=5,
    padding=10,
    pad_value=1
)

img = np.transpose(
    grid.numpy(),
    (1, 2, 0)
)

fig = plt.figure(figsize=(16, 16))
plt.axis("off")
plt.imshow(img);

### TensorBoard

After (or during) training, start TensorBoard from a terminal in this notebook's folder. `./runs` is the default logging directory of `SummaryWriter()`:

```bash
tensorboard --logdir ./runs
```

Then open the printed link (usually http://localhost:6006/) in the browser:
- **SCALARS** — `discriminator/gradients`, `discriminator/loss`, `generator/gradients`, `generator/loss` (progression of the loss throughout the training epochs)
- **IMAGES** — `generator/outputs`: the 5x5 grid of generated digits logged every 10 epochs (use the slider to scroll through the epochs)
- **GRAPHS** — the traced `Generator` / `Discriminator` graphs (double click a node, e.g. `Sequential[fcn]`, to see the detailed Linear / Dropout / Sigmoid layers)
- **TEXT** — the `WGAN` / `Init` text

Alternatively, TensorBoard can be shown inline in the notebook by running `%load_ext tensorboard` followed by `%tensorboard --logdir ./runs` in a new cell.

### Google Collaboratory

Notebook: https://colab.research.google.com/drive/13oGV5GNcnZVKw1zzsAEAH9nvms04uYnT

### Original Paper Code

Github: https://github.com/martinarjovsky/WassersteinGAN

### Bonus Tip

*Weight Update*

$$\omega_{t+1} = \omega_t + \alpha \frac{\partial \; J(\theta)}{\partial \omega}$$

*Chain Rule*

$$\frac{\partial \; J(\theta)}{\partial \omega} = \frac{\partial \; J(\theta)}{\partial \; output} \frac{\partial \; output}{\partial \omega}$$

*Derivative of Linear*

$$\frac{\partial \; output}{\partial \omega} = \frac{\partial \; (\omega x + b)}{\partial \omega} = x$$

*PyTorch Autograd*

```python
''' Instead of '''
loss.backward()

''' We did '''
output.backward(grad)

# fake
output.backward(-1)

# real
output.backward(+1)
```

### Coding Challenge

- Use both CGAN and WGAN together in one model
- Train the model on FashionMNIST dataset
- Instead of fully connected layers, try convolutional layers
- Play around with learning rate decay
- Try tensorboard histogram visualization of gradients

### Reading Assignment

- Conditional Generative Adversarial Nets, Mehdi Mirza, Simon Osindero, 2014 — https://arxiv.org/abs/1411.1784
- Wasserstein GAN, Martin Arjovsky, Soumith Chintala, Léon Bottou, 2017 — https://arxiv.org/abs/1701.07875